# Fine-tuning Llama-3.2-3B-Instruct on SAMSum with QLoRA

**Goal**: produce a LoRA adapter that turns the base instruction model into a strong dialogue summarizer, runnable on free Colab/Kaggle GPU.

**Recipe**: QLoRA = base model in 4-bit NF4 + LoRA rank-16 adapters on the attention projections, trained with `trl.SFTTrainer` using `assistant_only_loss=True`.

**Why this recipe?**
- 4-bit base (NF4 + double-quant) brings a 3B model to ~2 GB of VRAM. Fits on T4.
- LoRA on q/k/v/o is the standard sweet spot — small adapter, large quality gain.
- `SFTTrainer` is the canonical instruction-tuning trainer; `assistant_only_loss` ensures we don't waste loss on the prompt tokens.

**Time**: ~25 min on a T4 for 2 epochs over the full SAMSum train split (14.7k examples).

**Cost**: free if you run it in Colab.

## 0. Setup

If you're on Colab, uncomment the install cell. Locally, run `pip install -e ".[train]"` from the project root.

In [ ]:
# !pip install -q transformers>=4.46 peft>=0.13 trl>=0.12 accelerate>=1.1 bitsandbytes>=0.44 datasets>=3.1 \
#                rouge-score bert-score evaluate pydantic pydantic-settings

In [ ]:
# Llama-3.2 is a gated model — accept the license at https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct
# and log in here with a token that has read access.
from huggingface_hub import notebook_login
notebook_login()

## 1. Load and inspect SAMSum

SAMSum is ~16k crowd-written messenger conversations paired with short third-person summaries. We reformat each row into the Llama-3 chat template:

```
system: You are a concise dialogue-summarization assistant...
user:   Summarize this conversation:\n\n<dialogue>
assistant: <summary>
```

This matches the format the base model was instruction-tuned on, so fine-tuning is gentle — we're nudging the model to specialize, not reshaping how it talks.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data import load_samsum
ds = load_samsum()
print(ds)
print()
print('Example formatted row:')
for msg in ds['train'][0]['messages']:
    print(f"[{msg['role']}]\n{msg['content']}\n")

In [ ]:
# Quick stats: dialogue / summary length distribution.
import statistics
raw = load_samsum.__wrapped__ if hasattr(load_samsum, '__wrapped__') else None
from datasets import load_dataset
raw_train = load_dataset('samsum', split='train', trust_remote_code=True)
dia_lens = [len(x.split()) for x in raw_train['dialogue']]
sum_lens = [len(x.split()) for x in raw_train['summary']]
print(f"Dialogue words: median={statistics.median(dia_lens)}, p95={sorted(dia_lens)[int(0.95*len(dia_lens))]}")
print(f"Summary words:  median={statistics.median(sum_lens)}, p95={sorted(sum_lens)[int(0.95*len(sum_lens))]}")

**What to take away from this**: dialogues are long (median ~80 words, p95 ~200) but summaries are short (median ~20, p95 ~40). That's why we set `max_seq_length=1024` — it covers nearly all examples without padding most of them too aggressively, and `max_new_tokens=128` is plenty for generation.

## 2. Load the model with QLoRA

Three things happen inside `load_for_training()`:
1. `BitsAndBytesConfig` loads the base in 4-bit NF4 with double-quant.
2. `prepare_model_for_kbit_training` enables gradient checkpointing and fixes layer-norm dtypes (small things that bite if you skip them).
3. `get_peft_model` attaches the LoRA adapters and freezes the base.

The `print_trainable_parameters()` call at the end should show roughly **24M trainable / 3.2B total ≈ 0.75%**. That's the LoRA magic — you train less than 1% of the parameters and get nearly the quality of full fine-tuning.

In [ ]:
from src.model import load_for_training
model, tokenizer = load_for_training()

## 3. Train

We use `trl.SFTTrainer`. Key settings (in `src/train.py`):
- `assistant_only_loss=True` — loss is computed only on the assistant tokens. Critical: without this, the model is partly being graded on reproducing the prompt, which is wasted signal.
- `packing=False` — we keep one example per sequence so the loss mask is reliable.
- `bf16=True` and `gradient_checkpointing=True` — memory savings that combine to keep us under the T4's 16 GB.
- `lr=2e-4`, `cosine` schedule, `warmup_ratio=0.03` — standard LoRA recipe; LoRA tolerates higher learning rates than full fine-tuning.

For Colab, just run `python -m src.train --output_dir ./outputs/llama32-samsum-lora` in a terminal cell, or inline:

In [ ]:
from trl import SFTConfig, SFTTrainer
from src.config import settings

sft_config = SFTConfig(
    output_dir='./outputs/llama32-samsum-lora',
    num_train_epochs=settings.num_epochs,
    per_device_train_batch_size=settings.per_device_batch_size,
    per_device_eval_batch_size=settings.per_device_batch_size,
    gradient_accumulation_steps=settings.gradient_accumulation_steps,
    learning_rate=settings.learning_rate,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_seq_length=settings.max_seq_length,
    logging_steps=25,
    save_steps=500,
    eval_strategy='steps',
    eval_steps=500,
    save_total_limit=2,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    report_to='none',
    packing=False,
    assistant_only_loss=True,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds['train'],
    eval_dataset=ds['validation'],
    processing_class=tokenizer,
)
trainer.train()

**What you should see during training**:
- Train loss falls quickly from ~2 to ~1 within the first 200 steps, then plateaus near 0.9 by end of epoch 2.
- Eval loss tracks train loss closely (LoRA + SAMSum is hard to overfit at this rank).
- If you see eval loss starting to climb while train loss falls, lower `num_epochs` to 1.

In [ ]:
trainer.save_model('./outputs/llama32-samsum-lora')
tokenizer.save_pretrained('./outputs/llama32-samsum-lora')
print('Adapter saved. Size:')
!du -sh ./outputs/llama32-samsum-lora/

## 4. Sanity-check generation

Run one example through the adapter to confirm it summarizes (not just echoes).

In [ ]:
from src.infer import summarize

example = ('Hannah: Hey, do you have Betty\'s number?\n'
          'Amanda: Lemme check\n'
          'Amanda: Sorry, can\'t find it.\n'
          'Amanda: Ask Larry\n'
          'Amanda: He called her last time we were at the park together\n'
          'Hannah: I don\'t know him well\n'
          'Amanda: Just text him 🙂\n'
          'Hannah: Alright')

print(summarize(model, tokenizer, example))

## 5. Push the adapter to the Hub

Once you're happy with the metrics (see `02_evaluation_comparison.ipynb`), upload:

```bash
python scripts/push_to_hub.py \
    --adapter_dir ./outputs/llama32-samsum-lora \
    --repo_id <your-hf-username>/llama32-samsum-lora
```

That uploads the adapter (~30 MB) plus a generated model card. The Gradio app and HF Space will both load from this repo.